# Transformation

In diesem Notebook wird der bereinigte Datensatz in eine analysefreundliche Struktur überführt. Dabei werden Informationen, die sich aus den vorhandenen Daten
ableiten lassen, so aufbereitet, dass sie bei späteren Analysen nicht wiederholt berechnet werden müssen.

Dazu werden unter anderem Kalenderinformationen in einer separaten Datumstabelle bereitgestellt, zusätzliche Kennzahlen berechnet und die Daten in
Rechnungen und Rechnungspositionen aufgeteilt.

In [1]:
from paths import CLEANED_DATA_DIR, TRANSFORMED_DATA_DIR
import pandas as pd

dataset = pd.read_parquet(CLEANED_DATA_DIR / "online_retail_II.parquet")

## Datumstabelle

Zeitbezogene Analysen benötigen häufig Merkmale wie Jahr, Quartal, Monat, Kalenderwoche oder Wochentag. Diese Informationen lassen sich zwar jederzeit
aus **InvoiceDate** berechnen, müssten dann jedoch bei jeder Analyse erneut abgeleitet werden.

Daher wird eine separate Datumstabelle erstellt, die jedes im Datensatz vorkommende Datum genau einmal enthält und die zugehörigen Kalendermerkmale
zentral bereitstellt.

**YearQuarter** und **YearMonth** ermöglichen eine eindeutige chronologische Gruppierung nach Quartalen und Monaten über mehrere Jahre hinweg. Beide werden
als Datumswerte mit dem jeweiligen Periodenbeginn abgebildet.

Für Wochenanalysen werden **ISOYear** und **ISOWeek** verwendet. Das ISO-Jahr kann an Jahresgrenzen vom Kalenderjahr abweichen und ermöglicht zusammen
mit der ISO-Kalenderwoche eine eindeutige Zuordnung. Die Wochentage werden entsprechend der ISO-Konvention von Montag (`1`) bis Sonntag (`7`) nummeriert.

In [2]:
invoice_date = dataset["InvoiceDate"].dt
iso_calendar = invoice_date.isocalendar()

date_dataset = pd.DataFrame({
    "Date": invoice_date.normalize(),
    "Year": invoice_date.year,
    "Quarter": invoice_date.quarter,
    "YearQuarter": invoice_date.to_period("Q").dt.to_timestamp(),
    "Month": invoice_date.month,
    "YearMonth": invoice_date.to_period("M").dt.to_timestamp(),
    "MonthName": invoice_date.month_name(),
    "ISOYear": iso_calendar.year,
    "ISOWeek": iso_calendar.week,
    "Weekday": invoice_date.weekday + 1,
    "WeekdayName": invoice_date.day_name()
})

Jedes Datum kommt in der Datumstabelle nur einmal vor. **Date** wird daher als eindeutiger Schlüssel der Tabelle verwendet.

In [3]:
date_dataset = (
    date_dataset
    .drop_duplicates(subset="Date")
    .sort_values("Date")
    .set_index("Date")
)

## Umsatz je Rechnungsposition

**Price** beschreibt den Stückpreis eines Artikels und reicht daher allein nicht aus, um den Umsatz einer Rechnungsposition zu bestimmen. Dieser ergibt sich aus
der verkauften Menge und dem jeweiligen Stückpreis.

Da der Umsatz eine zentrale Größe für spätere Auswertungen nach beispielsweise Datum, Produkt, Rechnung oder Land ist, wird er bereits während der Transformation
als **Revenue** bereitgestellt.

In [4]:
dataset["Revenue"] = dataset["Quantity"] * dataset["Price"]

## Aufteilung des Rechnungszeitpunkts

**InvoiceDate** kombiniert zwei Informationen: das Kalenderdatum und die genaue Uhrzeit einer Rechnung.

Das Datum wird separat als **Date** bereitgestellt und dient gleichzeitig als Referenz auf die Datumstabelle. Dadurch steht das eigentliche Rechnungsdatum
direkt zur Verfügung, während zusätzliche Kalendermerkmale bei Bedarf über die Datumstabelle abgerufen werden können.

Die Uhrzeit bleibt als **Time** erhalten, damit auch Analysen innerhalb eines Tages möglich sind und die zeitliche Reihenfolge von Rechnungen weiterhin
nachvollzogen werden kann.

Daher wird **InvoiceDate** in **Date** und **Time** aufgeteilt.

In [5]:
dataset["Time"] = invoice_date.time
dataset["Date"] = invoice_date.normalize()

## Aufteilung in Rechnung und Rechnungsposition

Der Ausgangsdatensatz enthält die Eigenschaften einer Rechnung wiederholt für jede einzelne Rechnungsposition. Um diese unterschiedlichen Granularitäten
voneinander zu trennen, werden die Daten in eine Rechnungstabelle und eine Tabelle der Rechnungspositionen aufgeteilt.

### Rechnungstabelle

Die Tabelle **invoice** enthält die Informationen, die jeweils für die gesamte Rechnung gelten. **Invoice** ist dabei der eindeutige Schlüssel. **Date**
verweist gleichzeitig auf das entsprechende Datum in der Datumstabelle.

Zusätzlich werden mit **PositionCount**, **TotalQuantity** und **TotalRevenue** häufig benötigte Kennzahlen auf Rechnungsebene bereitgestellt.
Sie beschreiben die Anzahl der Rechnungspositionen, die gesamte Menge und den Gesamtumsatz einer Rechnung. Obwohl sich diese Werte aus den
Rechnungspositionen ableiten lassen, erleichtert ihre zusätzliche Bereitstellung spätere Analysen auf Rechnungsebene.

In [6]:
invoice_cols = [
    "Invoice",
    "Customer ID",
    "Country",
    "Date",
    "Time"
]

invoice_dataset = (
    dataset[invoice_cols]
    .drop_duplicates()
    .set_index("Invoice")
)

invoice_position_cols = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "Price",
    "Revenue"
]

invoice_position_dataset = dataset[invoice_position_cols]

invoice_dataset[["PositionCount", "TotalQuantity", "TotalRevenue"]] = (
    invoice_position_dataset
    .groupby("Invoice")
    .agg({
        "Invoice": "size",
        "Quantity": "sum",
        "Revenue": "sum"
    })
)

### Rechnungspositionstabelle

Die Tabelle **invoice_position** enthält die einzelnen Artikelpositionen einer Rechnung. Eine **Invoice** kann mehrere Positionen enthalten und ist daher
innerhalb dieser Tabelle nicht eindeutig.

Da der Ausgangsdatensatz keine Positionsnummer enthält, wird innerhalb jeder **Invoice** eine fortlaufende **Position** vergeben. Die Kombination
aus **Invoice** und **Position** identifiziert damit jede Rechnungsposition eindeutig. **Invoice** dient gleichzeitig als Fremdschlüssel zur
zugehörigen Rechnung.

In [7]:
invoice_position_dataset["Position"] = (
    invoice_position_dataset
    .groupby("Invoice")
    .cumcount() + 1
)

invoice_position_dataset = (
    invoice_position_dataset
    .set_index(["Invoice", "Position"])
)

### Vereinheitlichung der Produktbezeichnungen

Einzelne **StockCode**-Werte kommen im Ausgangsdatensatz mit unterschiedlichen **Description**-Werten vor. Da keine Information vorliegt, anhand derer eine
dieser Bezeichnungen fachlich als die korrekte bestimmt werden kann, wird für jeden **StockCode** eine einheitliche Bezeichnung festgelegt.

Dazu wird deterministisch die lexikografisch kleinste vorhandene **Description** verwendet. Dadurch bleibt **StockCode** weiterhin die eigentliche
Produktidentifikation, während **Description** für spätere Auswertungen und Visualisierungen als konsistente Anzeige-Bezeichnung verwendet werden kann.

In [8]:
description_by_stockcode = (
    invoice_position_dataset
    .groupby("StockCode")["Description"]
    .transform("min")
)

invoice_position_dataset["Description"] = description_by_stockcode

## Validierung des Datenmodells

Pandas erzwingt weder die Eindeutigkeit von Indizes noch die referenzielle Integrität zwischen DataFrames. Da die erzeugten Tabellen als relationales Datenmodell verwendet werden sollen, werden diese Eigenschaften abschließend überprüft.

Zunächst wird sichergestellt, dass die definierten Schlüssel der Tabellen eindeutig sind. Anschließend wird geprüft, ob alle Fremdschlüssel auf vorhandene Datensätze der jeweils referenzierten Tabelle verweisen.

In [9]:
assert date_dataset.index.is_unique
assert invoice_dataset.index.is_unique
assert invoice_position_dataset.index.is_unique

assert invoice_dataset["Date"].isin(date_dataset.index).all()

assert (
    invoice_position_dataset.index
    .get_level_values("Invoice")
    .isin(invoice_dataset.index)
    .all()
)

## Speichern der transformierten Daten

Nach erfolgreicher Validierung werden die drei Tabellen als Eingabe für die nachfolgenden Analysen im Parquet-Format gespeichert.

In [10]:
date_dataset.to_parquet(TRANSFORMED_DATA_DIR / "date.parquet")

invoice_dataset.to_parquet(TRANSFORMED_DATA_DIR / "invoice.parquet")

invoice_position_dataset.to_parquet(TRANSFORMED_DATA_DIR / "invoice_position.parquet")